# Mean +/- STD Oversampling: Bert

This notebook mirrors the baseline-versus-post-change pattern used in the feature-selection notebooks, but for augmentation.

It:
- loads the `bert` feature family
- evaluates baseline models on a held-out test split
- augments only the training split with class-conditional `mean - std`, `mean`, and `mean + std` synthetic rows
- evaluates the augmented training set against the same untouched test set
- saves richer artifacts for downstream visualization and audit


In [ ]:
from pathlib import Path
import json
import os
import sys
from contextlib import contextmanager

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from tqdm.auto import tqdm

plt.style.use('ggplot')
sns.set_theme(style='whitegrid')
np.random.seed(42)

print('Imports loaded.')


In [ ]:
CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / 'pyproject.toml').exists() else next(
    path for path in [CWD, *CWD.parents] if (path / 'pyproject.toml').exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from feature_augmentation.common import (
    DEFAULT_AUGMENTATION_METHOD,
    build_classification_report_frame,
    build_confusion_matrix_frame,
    build_stage_class_count_table,
    mean_std_oversample_training_frame,
    resolve_augmentation_artifact_dir,
    summarize_class_counts,
)
from feature_selection.common import (
    DEFAULT_EXCLUDED_EMOTIONS,
    DEFAULT_REQUIRE_AGREEMENT,
    METADATA_CANDIDATES,
    filter_feature_frame,
    include_xxx_from_env,
    machine_name_from_env,
    resolve_cpu_parallel_config,
    resolve_feature_source,
    resolve_random_forest_jobs,
    use_variant_artifact_dirs_from_env,
    variant_name,
)

DATASET_KEY = 'bert'
TARGET_COL = 'emotion'
TEST_SIZE = 0.20
RANDOM_STATE = 42
TARGET_PER_CLASS = 10_000
GROUP_SIZE = 5
AUGMENT_RANDOM_STATE = 42
USE_NROWS = None
REQUIRE_AGREEMENT = DEFAULT_REQUIRE_AGREEMENT
EXCLUDED_EMOTIONS = DEFAULT_EXCLUDED_EMOTIONS
AUGMENTATION_METHOD = DEFAULT_AUGMENTATION_METHOD

MACHINE_NAME = machine_name_from_env()
INCLUDE_XXX = include_xxx_from_env(default=True)
VARIANT_NAME = variant_name(INCLUDE_XXX)
USE_VARIANT_ARTIFACT_DIRS = use_variant_artifact_dirs_from_env(default=True)
MODEL_PARALLEL_JOBS, MODEL_PARALLEL_MODE = resolve_cpu_parallel_config(MACHINE_NAME)
RANDOM_FOREST_JOBS = resolve_random_forest_jobs(MACHINE_NAME)

source_path = resolve_feature_source(REPO_ROOT, DATASET_KEY)
artifact_dir = resolve_augmentation_artifact_dir(
    REPO_ROOT,
    include_xxx=INCLUDE_XXX,
    use_variant_dirs=USE_VARIANT_ARTIFACT_DIRS,
)

print(f'Repository root: {REPO_ROOT}')
print(f'Selected source: {source_path}')
print('Source exists:', source_path.exists())
print(f'Machine: {MACHINE_NAME}')
print(f'Include xxx: {INCLUDE_XXX} ({VARIANT_NAME})')
print(f'Artifact dir: {artifact_dir}')
print(f'CPU model parallel jobs: {MODEL_PARALLEL_JOBS} | mode={MODEL_PARALLEL_MODE}')
print(f'Augmentation method: {AUGMENTATION_METHOD}')


In [ ]:
if not source_path.exists():
    raise FileNotFoundError(f'Missing source CSV: {source_path}')

raw_df = pd.read_csv(source_path, nrows=USE_NROWS)

if TARGET_COL not in raw_df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found. Available: {list(raw_df.columns[:20])}...")

rows_loaded = len(raw_df)
df = filter_feature_frame(
    raw_df,
    target_col=TARGET_COL,
    include_xxx=INCLUDE_XXX,
    require_agreement=REQUIRE_AGREEMENT,
    excluded_emotions=EXCLUDED_EMOTIONS,
)

meta_cols = [c for c in df.columns if c in METADATA_CANDIDATES]
feature_cols = [c for c in df.columns if c not in METADATA_CANDIDATES]

raw_class_counts = summarize_class_counts(raw_df[TARGET_COL], label_name=TARGET_COL)
filtered_class_counts = summarize_class_counts(df[TARGET_COL], label_name=TARGET_COL)

print(f'Rows loaded: {rows_loaded}')
print(f'Rows after variant filtering: {len(df)}')
print(f'Metadata columns ({len(meta_cols)}): {meta_cols}')
print(f'Feature columns ({len(feature_cols)}).')
print('Sample feature columns:', feature_cols[:10])
print('\nRaw class distribution:')
print(raw_class_counts.to_string(index=False))
print('\nFiltered class distribution:')
print(filtered_class_counts.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(raw_class_counts[TARGET_COL], raw_class_counts['count'])
axes[0].set_title('Raw Class Distribution')
axes[0].set_xlabel(TARGET_COL)
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(filtered_class_counts[TARGET_COL], filtered_class_counts['count'])
axes[1].set_title(f'Filtered Class Distribution ({VARIANT_NAME})')
axes[1].set_xlabel(TARGET_COL)
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
model_df = df[[TARGET_COL] + feature_cols].copy()

rows_before = len(model_df)
model_df = model_df.dropna(subset=[TARGET_COL]).copy()
usable_feature_cols = [c for c in feature_cols if model_df[c].notna().all()]
dropped_feature_cols = [c for c in feature_cols if c not in usable_feature_cols]
model_df = model_df[[TARGET_COL] + usable_feature_cols].dropna(axis=0, how='any').copy()
rows_after = len(model_df)

X = model_df[usable_feature_cols].astype(float)
y = model_df[TARGET_COL].astype(str)
all_labels = sorted(y.unique())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

base_stage_class_counts_df = build_stage_class_count_table(
    {
        'raw_loaded': raw_df[TARGET_COL],
        'filtered': df[TARGET_COL],
        'model_ready': model_df[TARGET_COL],
        'train_reference': y_train,
        'test_reference': y_test,
    },
    label_name=TARGET_COL,
)

print(f'Rows before cleaning: {rows_before}')
print(f'Rows after cleaning:  {rows_after}')
print(f'Dropped rows:         {rows_before - rows_after}')
print(f'Feature columns used: {len(usable_feature_cols)}')
print(f'Dropped feature columns with NaNs: {len(dropped_feature_cols)}')
if dropped_feature_cols:
    print('Sample dropped feature columns:', dropped_feature_cols[:10])
print(f'Train shape: {X_train.shape}')
print(f'Test shape:  {X_test.shape}')
print(f'Classes: {all_labels}')

display(base_stage_class_counts_df)

plot_stage_order = ['filtered', 'train_reference', 'test_reference']
plot_df = base_stage_class_counts_df[base_stage_class_counts_df['stage'].isin(plot_stage_order)].copy()
plot_pivot = (
    plot_df
    .pivot(index=TARGET_COL, columns='stage', values='class_count')
    .fillna(0)
    .reindex(columns=plot_stage_order)
)

fig, ax = plt.subplots(figsize=(10, 5))
plot_pivot.plot(kind='bar', ax=ax)
ax.set_title('Reference Class Counts Used for Training and Evaluation')
ax.set_xlabel(TARGET_COL)
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
MODEL_SPECS = {
    'logreg': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=4000, class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
    'linear_svc': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LinearSVC(class_weight='balanced', random_state=RANDOM_STATE, max_iter=8000)),
    ]),
    'random_forest': Pipeline([
        ('scaler', 'passthrough'),
        ('clf', RandomForestClassifier(
            n_estimators=500,
            class_weight='balanced_subsample',
            random_state=RANDOM_STATE,
            n_jobs=RANDOM_FOREST_JOBS,
        )),
    ]),
}


@contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback

    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()


def _resolve_model_jobs(requested_jobs: int, n_models: int) -> int:
    if requested_jobs == -1:
        return max(1, min(os.cpu_count() or 1, n_models))
    return max(1, min(int(requested_jobs), n_models))


def _has_inner_parallelism(estimator) -> bool:
    params = estimator.get_params(deep=True)
    if 'clf__n_jobs' not in params:
        return False
    return params['clf__n_jobs'] not in (None, 1)


def _fit_predict_single_model(
    model_name: str,
    estimator,
    X_train_eval,
    y_train_eval,
    X_test_eval,
    y_test_eval,
    stage_name: str,
    force_single_core_inner_model: bool,
):
    model = clone(estimator)

    if force_single_core_inner_model and 'clf__n_jobs' in model.get_params(deep=True):
        model.set_params(clf__n_jobs=1)

    model.fit(X_train_eval, y_train_eval)
    y_pred = model.predict(X_test_eval)

    row = {
        'stage': stage_name,
        'model': model_name,
        'accuracy': accuracy_score(y_test_eval, y_pred),
        'precision_weighted': precision_score(y_test_eval, y_pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_test_eval, y_pred, average='weighted', zero_division=0),
        'f1_weighted': f1_score(y_test_eval, y_pred, average='weighted'),
        'f1_macro': f1_score(y_test_eval, y_pred, average='macro'),
    }
    return model_name, row, pd.Series(y_pred, index=y_test_eval.index)


def _evaluate_items_sequential(
    model_items,
    X_train_eval,
    y_train_eval,
    X_test_eval,
    y_test_eval,
    stage_name: str,
    desc: str,
    force_single_core_inner_model: bool,
):
    return [
        _fit_predict_single_model(
            model_name,
            estimator,
            X_train_eval,
            y_train_eval,
            X_test_eval,
            y_test_eval,
            stage_name,
            force_single_core_inner_model,
        )
        for model_name, estimator in tqdm(
            model_items,
            total=len(model_items),
            desc=desc,
            leave=False,
        )
    ]


def _evaluate_items_parallel(
    model_items,
    requested_jobs: int,
    X_train_eval,
    y_train_eval,
    X_test_eval,
    y_test_eval,
    stage_name: str,
    desc: str,
    force_single_core_inner_model: bool,
):
    if not model_items:
        return []

    n_jobs = _resolve_model_jobs(requested_jobs, len(model_items))
    if n_jobs == 1:
        return _evaluate_items_sequential(
            model_items,
            X_train_eval,
            y_train_eval,
            X_test_eval,
            y_test_eval,
            stage_name,
            desc,
            force_single_core_inner_model,
        )

    with tqdm_joblib(tqdm(total=len(model_items), desc=desc, leave=False)):
        return Parallel(n_jobs=n_jobs, backend='loky')(
            delayed(_fit_predict_single_model)(
                model_name,
                estimator,
                X_train_eval,
                y_train_eval,
                X_test_eval,
                y_test_eval,
                stage_name,
                force_single_core_inner_model,
            )
            for model_name, estimator in model_items
        )


def evaluate_models(X_train_eval, y_train_eval, X_test_eval, y_test_eval, stage_name: str):
    model_items = list(MODEL_SPECS.items())
    mode = str(MODEL_PARALLEL_MODE).strip().lower()
    if mode not in {'auto', 'outer', 'sequential'}:
        raise ValueError('MODEL_PARALLEL_MODE must be one of: auto, outer, sequential')

    requested_jobs = _resolve_model_jobs(MODEL_PARALLEL_JOBS, len(model_items))

    if mode == 'sequential' or requested_jobs == 1:
        evaluated = _evaluate_items_sequential(
            model_items,
            X_train_eval,
            y_train_eval,
            X_test_eval,
            y_test_eval,
            stage_name,
            desc=f'{stage_name}: fit/predict',
            force_single_core_inner_model=False,
        )
    elif mode == 'outer':
        evaluated = _evaluate_items_parallel(
            model_items,
            requested_jobs,
            X_train_eval,
            y_train_eval,
            X_test_eval,
            y_test_eval,
            stage_name,
            desc=f'{stage_name}: fit/predict',
            force_single_core_inner_model=True,
        )
    else:
        has_inner_parallel = any(_has_inner_parallelism(estimator) for _, estimator in model_items)
        evaluated = _evaluate_items_parallel(
            model_items,
            requested_jobs,
            X_train_eval,
            y_train_eval,
            X_test_eval,
            y_test_eval,
            stage_name,
            desc=f'{stage_name}: fit/predict',
            force_single_core_inner_model=has_inner_parallel,
        )

    rows = [row for _, row, _ in evaluated]
    preds = {model_name: pred_series for model_name, _, pred_series in evaluated}
    results_df = pd.DataFrame(rows).sort_values('f1_weighted', ascending=False, ignore_index=True)
    return results_df, preds


def plot_confusion_heatmaps(y_true, y_pred, title_prefix: str):
    labels = list(all_labels)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = confusion_matrix(y_true, y_pred, labels=labels, normalize='true')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
        ax=axes[0],
    )
    axes[0].set_title(f'{title_prefix} | Counts')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')

    sns.heatmap(
        cm_norm,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
        ax=axes[1],
        vmin=0.0,
        vmax=1.0,
    )
    axes[1].set_title(f'{title_prefix} | Row-normalized')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')

    plt.tight_layout()
    plt.show()


In [ ]:
baseline_results, baseline_preds = evaluate_models(
    X_train,
    y_train,
    X_test,
    y_test,
    stage_name='baseline',
)

print('Baseline metrics (all models):')
print(baseline_results.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

baseline_report_frames = []
baseline_confusion_frames = []
for model_name in tqdm(
    baseline_results['model'],
    total=len(baseline_results),
    desc='baseline: reports/confusion',
    leave=False,
):
    report_dict = classification_report(
        y_test,
        baseline_preds[model_name],
        labels=all_labels,
        target_names=all_labels,
        zero_division=0,
        output_dict=True,
    )
    print(f'\n[{model_name}] classification report (baseline):')
    print(classification_report(
        y_test,
        baseline_preds[model_name],
        labels=all_labels,
        target_names=all_labels,
        zero_division=0,
    ))
    baseline_report_frames.append(
        build_classification_report_frame(report_dict, stage_name='baseline', model_name=model_name)
    )
    baseline_confusion_frames.append(
        build_confusion_matrix_frame(
            confusion_matrix(y_test, baseline_preds[model_name], labels=all_labels),
            labels=all_labels,
            stage_name='baseline',
            model_name=model_name,
        )
    )

baseline_report_df = pd.concat(baseline_report_frames, ignore_index=True)
baseline_confusion_df = pd.concat(baseline_confusion_frames, ignore_index=True)

best_baseline_model = baseline_results.loc[0, 'model']
plot_confusion_heatmaps(y_test, baseline_preds[best_baseline_model], f'Baseline Confusion ({best_baseline_model})')

base_acc = float(baseline_results.loc[0, 'accuracy'])
base_f1 = float(baseline_results.loc[0, 'f1_weighted'])
base_f1_macro = float(baseline_results.loc[0, 'f1_macro'])


In [ ]:
X_train_aug, y_train_aug, synthetic_meta_df, augmentation_summary_df = mean_std_oversample_training_frame(
    X_train,
    y_train,
    target_per_class=TARGET_PER_CLASS,
    group_size=GROUP_SIZE,
    random_state=AUGMENT_RANDOM_STATE,
    sample_with_replacement=True,
)

train_counts_before = summarize_class_counts(y_train, label_name=TARGET_COL)
train_counts_after = summarize_class_counts(y_train_aug, label_name=TARGET_COL)
synthetic_only_labels = synthetic_meta_df[TARGET_COL] if TARGET_COL in synthetic_meta_df.columns else pd.Series(dtype=str)
synthetic_only_counts = summarize_class_counts(synthetic_only_labels, label_name=TARGET_COL)

stage_class_counts_df = build_stage_class_count_table(
    {
        'raw_loaded': raw_df[TARGET_COL],
        'filtered': df[TARGET_COL],
        'model_ready': model_df[TARGET_COL],
        'train_reference': y_train,
        'test_reference': y_test,
        'synthetic_only': synthetic_meta_df[TARGET_COL],
        'train_augmented': y_train_aug,
    },
    label_name=TARGET_COL,
)

counts_compare_df = train_counts_before.merge(
    train_counts_after,
    on=TARGET_COL,
    how='outer',
    suffixes=('_before', '_after'),
).fillna(0)
counts_compare_df[['count_before', 'count_after']] = counts_compare_df[['count_before', 'count_after']].astype(int)
counts_compare_df['synthetic_added'] = counts_compare_df['count_after'] - counts_compare_df['count_before']

if synthetic_meta_df.empty:
    synthetic_kind_counts_df = pd.DataFrame(columns=[TARGET_COL, 'synthetic_kind', 'count'])
else:
    synthetic_kind_counts_df = (
        synthetic_meta_df
        .groupby([TARGET_COL, 'synthetic_kind'], as_index=False)
        .size()
        .rename(columns={'size': 'count'})
    )

print(f'Original train shape:  {X_train.shape}')
print(f'Augmented train shape: {X_train_aug.shape}')
print(f'Synthetic rows added:  {len(synthetic_meta_df)}')
print('Augmentation summary by class:')
display(augmentation_summary_df)
print('Stage-by-stage class counts:')
display(stage_class_counts_df)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].bar(counts_compare_df[TARGET_COL], counts_compare_df['count_before'])
axes[0, 0].set_title('Training Class Distribution Before Augmentation')
axes[0, 0].set_xlabel(TARGET_COL)
axes[0, 0].set_ylabel('Count')
axes[0, 0].tick_params(axis='x', rotation=45)

axes[0, 1].bar(counts_compare_df[TARGET_COL], counts_compare_df['count_after'])
axes[0, 1].set_title('Training Class Distribution After Augmentation')
axes[0, 1].set_xlabel(TARGET_COL)
axes[0, 1].set_ylabel('Count')
axes[0, 1].tick_params(axis='x', rotation=45)

axes[1, 0].bar(counts_compare_df[TARGET_COL], counts_compare_df['synthetic_added'])
axes[1, 0].set_title('Synthetic Rows Added Per Emotion')
axes[1, 0].set_xlabel(TARGET_COL)
axes[1, 0].set_ylabel('Synthetic rows added')
axes[1, 0].tick_params(axis='x', rotation=45)

if synthetic_kind_counts_df.empty:
    axes[1, 1].text(0.5, 0.5, 'No synthetic rows generated', ha='center', va='center')
    axes[1, 1].set_axis_off()
else:
    kind_plot = synthetic_kind_counts_df.pivot(index=TARGET_COL, columns='synthetic_kind', values='count').fillna(0)
    kind_plot.plot(kind='bar', stacked=True, ax=axes[1, 1])
    axes[1, 1].set_title('Synthetic Kind Counts by Emotion')
    axes[1, 1].set_xlabel(TARGET_COL)
    axes[1, 1].set_ylabel('Synthetic rows')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].legend(title='Synthetic kind', loc='best')

plt.tight_layout()
plt.show()

stage_plot_order = ['train_reference', 'test_reference', 'synthetic_only', 'train_augmented']
stage_plot_df = stage_class_counts_df[stage_class_counts_df['stage'].isin(stage_plot_order)].copy()
stage_plot_pivot = (
    stage_plot_df
    .pivot(index=TARGET_COL, columns='stage', values='class_count')
    .fillna(0)
    .reindex(columns=stage_plot_order)
)

fig, ax = plt.subplots(figsize=(11, 5))
stage_plot_pivot.plot(kind='bar', ax=ax)
ax.set_title('Class Counts Across Reference, Synthetic, and Augmented Sets')
ax.set_xlabel(TARGET_COL)
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
augmented_results, augmented_preds = evaluate_models(
    X_train_aug,
    y_train_aug,
    X_test,
    y_test,
    stage_name='augmented',
)

print('Post-augmentation metrics (all models):')
print(augmented_results.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

augmented_report_frames = []
augmented_confusion_frames = []
for model_name in tqdm(
    augmented_results['model'],
    total=len(augmented_results),
    desc='augmented: reports/confusion',
    leave=False,
):
    report_dict = classification_report(
        y_test,
        augmented_preds[model_name],
        labels=all_labels,
        target_names=all_labels,
        zero_division=0,
        output_dict=True,
    )
    print(f'\n[{model_name}] classification report (post-augmentation):')
    print(classification_report(
        y_test,
        augmented_preds[model_name],
        labels=all_labels,
        target_names=all_labels,
        zero_division=0,
    ))
    augmented_report_frames.append(
        build_classification_report_frame(report_dict, stage_name='augmented', model_name=model_name)
    )
    augmented_confusion_frames.append(
        build_confusion_matrix_frame(
            confusion_matrix(y_test, augmented_preds[model_name], labels=all_labels),
            labels=all_labels,
            stage_name='augmented',
            model_name=model_name,
        )
    )

augmented_report_df = pd.concat(augmented_report_frames, ignore_index=True)
augmented_confusion_df = pd.concat(augmented_confusion_frames, ignore_index=True)

best_augmented_model = augmented_results.loc[0, 'model']
plot_confusion_heatmaps(y_test, augmented_preds[best_augmented_model], f'Augmented Confusion ({best_augmented_model})')

comparison_df = baseline_results[['model', 'accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted', 'f1_macro']].merge(
    augmented_results[['model', 'accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted', 'f1_macro']],
    on='model',
    suffixes=('_baseline', '_augmented'),
)
comparison_df['accuracy_delta'] = comparison_df['accuracy_augmented'] - comparison_df['accuracy_baseline']
comparison_df['precision_weighted_delta'] = comparison_df['precision_weighted_augmented'] - comparison_df['precision_weighted_baseline']
comparison_df['recall_weighted_delta'] = comparison_df['recall_weighted_augmented'] - comparison_df['recall_weighted_baseline']
comparison_df['f1_weighted_delta'] = comparison_df['f1_weighted_augmented'] - comparison_df['f1_weighted_baseline']
comparison_df['f1_macro_delta'] = comparison_df['f1_macro_augmented'] - comparison_df['f1_macro_baseline']
comparison_df = comparison_df.sort_values('f1_weighted_augmented', ascending=False, ignore_index=True)

print('\nMetric delta by model (augmentation - baseline):')
print(comparison_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

aug_acc = float(augmented_results.loc[0, 'accuracy'])
aug_f1 = float(augmented_results.loc[0, 'f1_weighted'])
aug_f1_macro = float(augmented_results.loc[0, 'f1_macro'])

fig, ax = plt.subplots(figsize=(10, 5))
metric_delta_plot = comparison_df.set_index('model')[
    ['accuracy_delta', 'precision_weighted_delta', 'recall_weighted_delta', 'f1_weighted_delta', 'f1_macro_delta']
]
metric_delta_plot.plot(kind='bar', ax=ax)
ax.axhline(0.0, color='black', linewidth=1)
ax.set_title('Metric Delta by Model (Augmented - Baseline)')
ax.set_xlabel('Model')
ax.set_ylabel('Metric delta')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

best_models = [best_baseline_model, best_augmented_model]
best_report_compare_df = (
    pd.concat([baseline_report_df, augmented_report_df], ignore_index=True)
    .query("label_type == 'class' and model in @best_models")
    .copy()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
precision_plot_df = best_report_compare_df.pivot_table(
    index='label',
    columns='stage',
    values='precision',
    aggfunc='first',
).fillna(0.0)
precision_plot_df.plot(kind='bar', ax=axes[0])
axes[0].set_title('Per-class Precision: Best Baseline vs Best Augmented')
axes[0].set_xlabel(TARGET_COL)
axes[0].set_ylabel('Precision')
axes[0].tick_params(axis='x', rotation=45)

f1_plot_df = best_report_compare_df.pivot_table(
    index='label',
    columns='stage',
    values='f1_score',
    aggfunc='first',
).fillna(0.0)
f1_plot_df.plot(kind='bar', ax=axes[1])
axes[1].set_title('Per-class F1: Best Baseline vs Best Augmented')
axes[1].set_xlabel(TARGET_COL)
axes[1].set_ylabel('F1 score')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
out_dir = artifact_dir
out_dir.mkdir(parents=True, exist_ok=True)

train_aug_path = out_dir / f'{DATASET_KEY}_X_train_augmented.csv'
train_aug_labels_path = out_dir / f'{DATASET_KEY}_y_train_augmented.csv'
test_ref_path = out_dir / f'{DATASET_KEY}_X_test_reference.csv'
test_ref_labels_path = out_dir / f'{DATASET_KEY}_y_test_reference.csv'
synthetic_meta_path = out_dir / f'{DATASET_KEY}_synthetic_metadata.csv'
summary_path = out_dir / f'{DATASET_KEY}_augmentation_summary.csv'
class_counts_path = out_dir / f'{DATASET_KEY}_class_counts.csv'
synthetic_kind_counts_path = out_dir / f'{DATASET_KEY}_synthetic_kind_counts.csv'
baseline_metrics_path = out_dir / f'{DATASET_KEY}_baseline_model_metrics.csv'
augmented_metrics_path = out_dir / f'{DATASET_KEY}_augmented_model_metrics.csv'
comparison_path = out_dir / f'{DATASET_KEY}_model_metric_deltas.csv'
classification_reports_path = out_dir / f'{DATASET_KEY}_classification_reports.csv'
confusion_matrices_path = out_dir / f'{DATASET_KEY}_confusion_matrices.csv'
prediction_path = out_dir / f'{DATASET_KEY}_test_predictions.csv'
meta_path = out_dir / f'{DATASET_KEY}_run_metadata.json'

X_train_aug.to_csv(train_aug_path, index=False)
y_train_aug.to_frame(name=TARGET_COL).to_csv(train_aug_labels_path, index=False)
X_test.to_csv(test_ref_path, index=False)
y_test.to_frame(name=TARGET_COL).to_csv(test_ref_labels_path, index=False)
synthetic_meta_df.to_csv(synthetic_meta_path, index=False)
augmentation_summary_df.to_csv(summary_path, index=False)
stage_class_counts_df.to_csv(class_counts_path, index=False)
synthetic_kind_counts_df.to_csv(synthetic_kind_counts_path, index=False)
baseline_results.to_csv(baseline_metrics_path, index=False)
augmented_results.to_csv(augmented_metrics_path, index=False)
comparison_df.to_csv(comparison_path, index=False)

classification_reports_df = pd.concat([baseline_report_df, augmented_report_df], ignore_index=True)
classification_reports_df.to_csv(classification_reports_path, index=False)

confusion_matrices_df = pd.concat([baseline_confusion_df, augmented_confusion_df], ignore_index=True)
confusion_matrices_df.to_csv(confusion_matrices_path, index=False)

prediction_df = pd.DataFrame({
    'test_index': X_test.index.astype(int),
    TARGET_COL: y_test.reset_index(drop=True),
})
for model_name in baseline_results['model']:
    prediction_df[f'baseline__{model_name}'] = baseline_preds[model_name].reset_index(drop=True)
for model_name in augmented_results['model']:
    prediction_df[f'augmented__{model_name}'] = augmented_preds[model_name].reset_index(drop=True)
prediction_df.to_csv(prediction_path, index=False)

baseline_records = baseline_results.to_dict(orient='records')
augmented_records = augmented_results.to_dict(orient='records')
comparison_records = comparison_df.to_dict(orient='records')
for collection in (baseline_records, augmented_records, comparison_records):
    for row in collection:
        for key, value in list(row.items()):
            if isinstance(value, (np.floating, float)):
                row[key] = float(value)

run_meta = {
    'dataset_key': DATASET_KEY,
    'source_path': str(source_path),
    'artifact_dir': str(out_dir),
    'target_col': TARGET_COL,
    'machine_name': MACHINE_NAME,
    'include_xxx': bool(INCLUDE_XXX),
    'variant_name': VARIANT_NAME,
    'require_agreement': bool(REQUIRE_AGREEMENT),
    'excluded_emotions': list(EXCLUDED_EMOTIONS),
    'use_variant_artifact_dirs': bool(USE_VARIANT_ARTIFACT_DIRS),
    'rows_loaded': int(rows_loaded),
    'rows_after_filter': int(len(df)),
    'rows_used': int(len(model_df)),
    'train_rows_reference': int(len(X_train)),
    'test_rows_reference': int(len(X_test)),
    'original_features': int(X_train.shape[1]),
    'feature_columns_used': int(len(usable_feature_cols)),
    'dropped_feature_columns': int(len(dropped_feature_cols)),
    'classes': list(all_labels),
    'random_state': RANDOM_STATE,
    'test_size': TEST_SIZE,
    'model_parallel_jobs': int(MODEL_PARALLEL_JOBS),
    'model_parallel_mode': MODEL_PARALLEL_MODE,
    'augmentation_method': AUGMENTATION_METHOD,
    'augmentation_target_per_class': int(TARGET_PER_CLASS),
    'augmentation_group_size': int(GROUP_SIZE),
    'augmentation_random_state': int(AUGMENT_RANDOM_STATE),
    'synthetic_rows_added': int(len(synthetic_meta_df)),
    'train_rows_before_augmentation': int(len(X_train)),
    'train_rows_after_augmentation': int(len(X_train_aug)),
    'best_baseline_model': str(best_baseline_model),
    'best_augmented_model': str(best_augmented_model),
    'baseline_accuracy': float(base_acc),
    'baseline_f1_weighted': float(base_f1),
    'baseline_f1_macro': float(base_f1_macro),
    'augmented_accuracy': float(aug_acc),
    'augmented_f1_weighted': float(aug_f1),
    'augmented_f1_macro': float(aug_f1_macro),
    'baseline_model_metrics': baseline_records,
    'augmented_model_metrics': augmented_records,
    'model_metric_deltas': comparison_records,
    'artifact_files': {
        'train_augmented': str(train_aug_path),
        'train_augmented_labels': str(train_aug_labels_path),
        'test_reference': str(test_ref_path),
        'test_reference_labels': str(test_ref_labels_path),
        'synthetic_metadata': str(synthetic_meta_path),
        'augmentation_summary': str(summary_path),
        'class_counts': str(class_counts_path),
        'synthetic_kind_counts': str(synthetic_kind_counts_path),
        'baseline_model_metrics': str(baseline_metrics_path),
        'augmented_model_metrics': str(augmented_metrics_path),
        'model_metric_deltas': str(comparison_path),
        'classification_reports': str(classification_reports_path),
        'confusion_matrices': str(confusion_matrices_path),
        'test_predictions': str(prediction_path),
    },
}

with meta_path.open('w', encoding='utf-8') as f:
    json.dump(run_meta, f, indent=2)

print('Saved artifacts:')
print('-', train_aug_path)
print('-', train_aug_labels_path)
print('-', test_ref_path)
print('-', test_ref_labels_path)
print('-', synthetic_meta_path)
print('-', summary_path)
print('-', class_counts_path)
print('-', synthetic_kind_counts_path)
print('-', baseline_metrics_path)
print('-', augmented_metrics_path)
print('-', comparison_path)
print('-', classification_reports_path)
print('-', confusion_matrices_path)
print('-', prediction_path)
print('-', meta_path)


In [ ]:
delta_cols = [
    'model',
    'accuracy_baseline',
    'accuracy_augmented',
    'accuracy_delta',
    'precision_weighted_baseline',
    'precision_weighted_augmented',
    'precision_weighted_delta',
    'recall_weighted_baseline',
    'recall_weighted_augmented',
    'recall_weighted_delta',
    'f1_weighted_baseline',
    'f1_weighted_augmented',
    'f1_weighted_delta',
    'f1_macro_baseline',
    'f1_macro_augmented',
    'f1_macro_delta',
]

delta_table = comparison_df[delta_cols].copy().sort_values('f1_weighted_delta', ascending=False, ignore_index=True)

print('Delta table (augmentation - baseline):')
print(delta_table.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

best_gain_row = delta_table.iloc[0]
print(
    f"\nBest weighted-F1 delta after augmentation: {best_gain_row['model']} | "
    f"Delta F1-weighted={best_gain_row['f1_weighted_delta']:+.4f}, "
    f"Delta Macro-F1={best_gain_row['f1_macro_delta']:+.4f}, "
    f"Delta Accuracy={best_gain_row['accuracy_delta']:+.4f}"
)
